In [2]:
import manim as mn
from manim import *

import numpy as np

config.media_width = "75%"
config.verbosity = "WARNING"

print(mn.__version__)

0.20.1


In [48]:
%%manim -ql DerivingFiberglassWidth

# i want to show how layering thickness onto a circle increases the circumference. so i want to take it in steps 
# 1. start with the final derived equation =PI() * (N * D + T * (N - 1) * N) N = num wraps, D = starting diameter, T = thickness per wrap. resulting value = total sum of circumferences 
# 2. show an example for the equation (plug in example values and get example result) 
# 3. draw circles: starting diameter ) layer ) layer ) ... N layers ) outer diameter 
# 4. (may want to save state before this) "zoom in" transform the circles so that the right edge of the starting diameter is on the left of the screen and the outer diameter is near the right edge of the screen, such that the wraps take up most of the screen. labels pop up showing that each layer is the same thickness 
# 5. zoom back out (may want to use a restore state), then upwrap the circles to lines show that each layer has a different circumference. 
# 6. write a simple equation to show calculating the circumference for each layer 
# 7. turn that simple equation into a summation equation 
# 8. simplify the summation equation into the telescoping series for the final equation of total circumferences


class DerivingFiberglassWidth(Scene):
    def construct(self):
        # Step 1: Show the final derived equation
        final_equation = MathTex(r"C = N \cdot \pi  \left( D + T \cdot (N - 1) \right)")
        self.play(Write(final_equation))
        self.wait(2)
        self.play(FadeOut(final_equation))

        # Step 2: Show an example for the equation
        example_values = MathTex(r"N = 4,\: D = 2,\: T = 1")
        example_result = MathTex(r"C = 4 \pi \left( 2 + 1 \cdot (4 - 1) \right) = 4 \pi (2 + 3) = 20\pi")
        self.play(Write(example_values))
        self.wait(1)
        self.play(Transform(example_values, example_result))
        self.wait(2)
        self.play(FadeOut(example_values))

        # Step 3: Draw circles for each layer
        scale = 0.5
        starting_diameter = Circle(radius=1 * scale, color=GRAY)
        layers = VGroup(*[Circle(radius=(1 + i * 2) * scale, color=BLUE) for i in range(1, 5)])  # 4 layers
        all_circles = VGroup(starting_diameter, *layers)
        
        self.play(Create(starting_diameter))
        for layer in layers:
            self.play(Create(layer))
            self.wait(0.5)

        # Step 4: Zoom in and show thickness
        zoomed_circles = all_circles.copy()
        zoomed_circles.scale(0.5).to_edge(RIGHT)
        
        thickness_labels = VGroup(*[MathTex(f"T_{i+1}").next_to(layers[i], LEFT) for i in range(len(layers))])
        
        self.play(Transform(all_circles, zoomed_circles))
        self.play(FadeIn(thickness_labels))
        self.wait(2)

        # Step 5: Zoom back out and unwrap circles to lines

        # calculate each layer's circumference and represent it as a line
        circumferences = [np.pi * (2 + i * 2) * scale for i in range(len(layers))]

        # generate lines representing each layer's circumference

        unwrap_scale = 1

        unwrapped_lines = VGroup(*[Line(start=LEFT, end=RIGHT * circumferences[i] * unwrap_scale).next_to(layers[i], LEFT) for i in range(len(layers))])

        # make each line spaced vertically
        for i, line in enumerate(unwrapped_lines):
            line.to_edge(UP)
            line.shift(DOWN * (i+1) * 1.5)
            line.to_edge(RIGHT)

        
        self.play(FadeOut(thickness_labels))
        self.play(Transform(all_circles, unwrapped_lines))
        self.wait(2)

        # Step 6: Write a simple equation for each layer's circumference
        layer_equations = VGroup(*[MathTex(f"C_{i+1}", r"= \pi \cdot (D + ", f"{i+1}", r" \cdot 2T)").next_to(unwrapped_lines[i], UP) for i in range(len(layers))])
        # set equation 1 to align to right edge of the screen for later smoother transform
        layer_equations[0].to_edge(RIGHT)
        self.play(Write(layer_equations))
        self.wait(2)

        # hide lines and layer equations
        self.play(FadeOut(all_circles))
        # move all layer equations to the right
        for i, eq in enumerate(layer_equations):
            self.play(eq.animate.to_edge(RIGHT),run_time=0.25)
        # self.play(FadeOut(layer_equations)) # could actually be cooler to transform these into the total circumference equation

        # 6.5: write out all sums into one line
        total_circumference_equation = MathTex(
            "C_{total}", "=",
            "C_1", "+\\\\",
            "C_2", "+\\\\",
            "C_3", "+\\\\",
            "C_4"
        ).to_edge(LEFT)
        self.play(Write(total_circumference_equation))
        self.wait(2)

        multi_line_equation = MathTex(r"C_{total} = \pi \cdot (D + 1 \cdot 2T) + \\ \pi \cdot (D + 2 \cdot 2T) + \\ \pi \cdot (D + 3 \cdot 2T) + \\ \pi \cdot (D + 4 \cdot 2T)")
                
        # transform group: total_circumference_equation and all layer_equations turn into the multi line equation
        # transform_group = VGroup(total_circumference_equation, *layer_equations)
        # self.play(TransformMatchingTex(transform_group, multi_line_equation))
        # self.play(Transform(layer_equations, multi_line_equation))
        self.play(FadeOut(layer_equations))
        self.play(TransformMatchingTex(total_circumference_equation, multi_line_equation))
        self.wait(2)

        delay = 0.5

        # Step 7: Turn that simple equation into a summation equation
        summation_equation = MathTex(                   "C_{total} =",  r" \sum_{i=1}^{N}",   r" \pi",    r" (D + i \cdot 2T)")
        self.play(TransformMatchingTex(multi_line_equation, summation_equation))
        self.wait(delay)

        standard_summation_equation = MathTex(          "C_{total} =",  r" \sum_{i=0}^{N-1}", r" \pi ",   r" (D + i \cdot 2T)")
        self.play(TransformMatchingTex(summation_equation, standard_summation_equation))
        self.wait(delay)

        # factor out pi
        factored_summation_equation = MathTex(          "C_{total} =",  r" \pi ",r"\sum_{i=0}^{N-1}",r" (D + i \cdot 2T)")
        self.play(TransformMatchingTex(standard_summation_equation, factored_summation_equation))
        self.wait(delay)

        # split summation into pi (D) + pi (2T * summation)
        split_summation_equation = MathTex(             "C_{total} =",r" \pi ",r"\left( \sum_{i=0}^{N-1}",r" D + 2T",r" \sum_{i=0}^{N-1} i \right)")
        self.play(TransformMatchingTex(factored_summation_equation, split_summation_equation))
        self.wait(delay)

        # evaluate left side of the summation: sum_{i=0}^{N-1} D = N * D
        evaluated_left_summation_equation = MathTex(    "C_{total} =",r" \pi",r" \left( N \cdot ",r"D + 2T",r" \sum_{i=0}^{N-1} i \right)")
        self.play(TransformMatchingTex(split_summation_equation, evaluated_left_summation_equation))
        self.wait(delay)

        # evaluate right side of the summation: sum_{i=0}^{N-1} i = (N-1) * N / 2
        evaluated_right_summation_equation = MathTex(   "C_{total} =",r" \pi",r" \left( N \cdot",r" D",r" +",r" 2",r"T",r" \cdot \frac{",r"(N-1) \cdot N",r"}{2} ",r"\right)")
        self.play(TransformMatchingTex(evaluated_left_summation_equation, evaluated_right_summation_equation))
        self.wait(delay)

        # # move 2T to the top of the fraction
        # moved_2T_summation_equation = MathTex(          "C_{total} =",r" \pi",r" \left( N \cdot",r" D",r" +",r" \frac{ ",r"2",r"T \cdot (N-1) \cdot N",r" }{2} ",r"\right)")
        # self.play(TransformMatchingTex(evaluated_right_summation_equation, moved_2T_summation_equation))
        # self.wait(delay)

        # cancel out the 2's on the right side of the summation
        canceled_summation_equation = MathTex(          "C_{total} =",r" \pi ",r"\left( N \cdot",r" D ",r"+",r" T ",r"\cdot",r" (N-1) \cdot N",r" \right)")
        self.play(TransformMatchingTex(evaluated_right_summation_equation, canceled_summation_equation))
        self.wait(delay)

        # Step 8: Simplify the summation equation into the telescoping series
        telescoping_equation = MathTex(                 r"C_{total} =",r" N",r" \cdot",r" \pi",r" \left(",r" D",r" + T",r" \cdot (N - 1)",r"\right)")
        self.play(TransformMatchingTex(canceled_summation_equation, telescoping_equation))
        self.wait(2)

Manim Community v0.20.1

In [48]:
%%manim -ql CuttingSheets

# in this animation:
# 1. draw all rectangles for sheets to be cut (mold release, fiberglass, outer peel ply)
# 2. animate each rectangle wrapping around a cylinder (mandrel) 
# 3. remove the inner cylinder (mandrel)
# 4. remove the inner most layer (mold release)
# 6. remove the outer peel ply


# 1. draw all rectangles
# example dimensions:
# mold release 533.6mm x 151.238mm
# fiberglass 482.8mm x 492.275mm
# outer peel ply 457.4mm x 157.576mm
# label each rectangle inside with the type of sheet, label one edge edge for each dimension (1 X and 1 Y for each rectangle)



class CuttingSheets(Scene):
    def construct(self):
        scale = 0.5
        # Create rectangles for each sheet
        mold_release = Rectangle(width=5.336*scale, height=1.51238*scale, color=BLUE, fill_opacity=0.5)
        fiberglass = Rectangle(width=4.828*scale, height=4.92275*scale, color=GREEN, fill_opacity=0.5)
        outer_peel_ply = Rectangle(width=4.574*scale, height=1.57576*scale, color=RED, fill_opacity=0.5)

        # Position the rectangles
        mold_release.to_edge(UP)
        mold_release.shift(DOWN)
        fiberglass.next_to(mold_release, DOWN, buff=1)
        outer_peel_ply.next_to(fiberglass, DOWN, buff=1)

        # Add labels for each rectangle
        mold_release_label = Text("Mold Release", color=BLUE).next_to(mold_release.get_right(), RIGHT)
        fiberglass_label = Text("Fiberglass", color=GREEN).next_to(fiberglass.get_right(), RIGHT)
        outer_peel_ply_label = Text("Outer Peel Ply", color=RED).next_to(outer_peel_ply.get_right(), RIGHT)

        # Add dimension labels for each rectangle
        mold_release_dim_x = Text("53.36cm", color=BLUE).next_to(mold_release.get_left(), LEFT)
        mold_release_dim_y = Text("15.13cm", color=BLUE).next_to(mold_release.get_top(), UP)

        fiberglass_dim_x = Text("48.28cm", color=GREEN).next_to(fiberglass.get_left(), LEFT)
        fiberglass_dim_y = Text("49.23cm", color=GREEN).next_to(fiberglass.get_top(), UP)

        outer_peel_ply_dim_x = Text("45.74cm", color=RED).next_to(outer_peel_ply.get_left(), LEFT)
        outer_peel_ply_dim_y = Text("15.76cm", color=RED).next_to(outer_peel_ply.get_top(), UP)

        # Animate the creation of rectangles and labels
        self.play(Create(mold_release), Write(mold_release_label), Write(mold_release_dim_x), Write(mold_release_dim_y))
        self.wait()
        self.play(Create(fiberglass), Write(fiberglass_label), Write(fiberglass_dim_x), Write(fiberglass_dim_y))
        self.wait()
        self.play(Create(outer_peel_ply), Write(outer_peel_ply_label), Write(outer_peel_ply_dim_x), Write(outer_peel_ply_dim_y))

        self.wait(2)

        # fade out all labels
        self.play(
            FadeOut(mold_release_label),
            FadeOut(fiberglass_label),
            FadeOut(outer_peel_ply_label),
            FadeOut(mold_release_dim_x),
            FadeOut(mold_release_dim_y),
            FadeOut(fiberglass_dim_x),
            FadeOut(fiberglass_dim_y),
            FadeOut(outer_peel_ply_dim_x),
            FadeOut(outer_peel_ply_dim_y)
        )


        # set rectangle opacities to 0.9
        # mold_release.set_fill(opacity=0.9)
        # fiberglass.set_fill(opacity=0.9)
        # outer_peel_ply.set_fill(opacity=0.9)



        # overlap the rectangles. have the mold release on the bottom, fiberglass in the middle, and outer peel ply on top.
        mold_release.set_z_index(-1)
        fiberglass.set_z_index(0)
        outer_peel_ply.set_z_index(1)
        self.play(
            mold_release.animate.move_to(ORIGIN),
            fiberglass.animate.move_to(ORIGIN),
            outer_peel_ply.animate.move_to(ORIGIN),
        )

        # "zoom in" on the right edge of the fiberglass rectangle by moving everything to the left by 1/2 the width of the fiberglass and scaling all rectangles by scale
        scale = 4

        self.play(
            mold_release.animate.scale(scale),
            fiberglass.animate.scale(scale),
            outer_peel_ply.animate.scale(scale))

        # save state of rectangles
        mold_release.save_state()
        fiberglass.save_state()
        outer_peel_ply.save_state()

        self.play(
            mold_release.animate.shift(LEFT * fiberglass.width / 2),
            fiberglass.animate.shift(LEFT * fiberglass.width / 2),
            outer_peel_ply.animate.shift(LEFT * fiberglass.width / 2)
        )

        self.wait(3)

        self.play(Restore(mold_release), Restore(fiberglass), Restore(outer_peel_ply))


        # align peel ply and mold release upper edge to the upper edge of fiberglass


        # scale again and move everything down by 1/2 the height of the fiberglass rectangle

        
        # set fiberglass height to 1/4 of original height
        fiberglass.set(height=fiberglass.height/4, width=fiberglass.width)
        short_fiberglass = Rectangle(width=fiberglass.width, height=fiberglass.height/4, color=GREEN, fill_opacity=0.5).to_edge(UP)
        self.play(
            Transform(fiberglass, short_fiberglass)
        )
        self.play(
            AnimationGroup(
            mold_release.animate.align_to(fiberglass, UP),
            outer_peel_ply.animate.align_to(fiberglass, UP),
            lag_ratio=0.1
            )
        )
        

        self.wait(3)

Manim Community v0.20.1

In [14]:
%%manim -ql WrappingProcess



class WrappingProcess(ThreeDScene):

    def construct(self):

        ###############################################################
        # Parameters
        ###############################################################

        scale = 0.5
        wait_time = 0.3

        mandrel_radius = 0.38

        mold_release_thickness = 0.04
        fiberglass_thickness   = 0.16
        peel_ply_thickness     = 0.04

        ###############################################################
        # Camera
        ###############################################################

        self.set_camera_orientation(
            phi=65 * DEGREES,
            theta=-55 * DEGREES,
        )

        ###############################################################
        # Flat sheets
        ###############################################################

        mold_release = Rectangle(
            width=1.51238*scale,
            height=5.336*scale,
            color=BLUE,
            fill_opacity=.6,
        )

        fiberglass = Rectangle(
            width=4.92275*scale,
            height=4.828*scale,
            color=GREEN,
            fill_opacity=.6,
        )

        peel_ply = Rectangle(
            width=1.57576*scale,
            height=4.574*scale,
            color=RED,
            fill_opacity=.6,
        )

        sheets = VGroup(
            mold_release,
            fiberglass,
            peel_ply
        ).arrange(RIGHT, buff=1.2)

        self.play(Create(sheets))
        self.wait()

        ###############################################################
        # Compress into strips
        ###############################################################

        # self.play(

        #     mold_release.animate.stretch(0.02, dim=1),

        #     fiberglass.animate.stretch(0.02, dim=1),

        #     peel_ply.animate.stretch(0.02, dim=1),

        #     run_time=1.5
        # )

        # self.wait()

        ###############################################################
        # Move strips beside mandrel
        ###############################################################

        center = LEFT*3

        self.play(
            sheets.animate.arrange(DOWN, buff=.4).move_to(RIGHT*2)
        )

        ###############################################################
        # Mandrel
        ###############################################################

        mandrel = Cylinder(
            radius=mandrel_radius,
            height=2.8,
            resolution=32,
            fill_color=GRAY,
            fill_opacity=1,
        )

        # rotate mandrel to be horizontal
        mandrel.rotate(90 * DEGREES, axis=RIGHT)

        mandrel.move_to(center)

        self.play(Create(mandrel))

        ###############################################################
        # Wrap helper
        ###############################################################

        def make_layer(radius, color):

            return Cylinder(
                radius=radius,
                height=2.8,
                resolution=48,
                fill_color=color,
                fill_opacity=.35,
                stroke_color=color,
            ).move_to(center).rotate(90 * DEGREES, axis=RIGHT)

        ###############################################################
        # Mold release
        ###############################################################

        mold_layer = make_layer(
            mandrel_radius + mold_release_thickness,
            BLUE,
        )

        self.play(
            ReplacementTransform(
                mold_release,
                mold_layer,
            ),
            run_time=2,
        )

        # self.play(
        #     Rotate(
        #         mold_layer,
        #         angle=TAU,
        #         axis=OUT,
        #         about_point=center,
        #     ),
        #     run_time=2,
        # )

        ###############################################################
        # Fiberglass
        ###############################################################

        glass_layer = make_layer(
            mandrel_radius
            + mold_release_thickness
            + fiberglass_thickness,
            GREEN,
        )

        self.play(
            ReplacementTransform(
                fiberglass,
                glass_layer,
            ),
            run_time=2,
        )

        # self.play(
        #     Rotate(
        #         glass_layer,
        #         angle=TAU,
        #         axis=OUT,
        #         about_point=center,
        #     ),
        #     run_time=2,
        # )

        ###############################################################
        # Peel ply
        ###############################################################

        peel_layer = make_layer(
            mandrel_radius
            + mold_release_thickness
            + fiberglass_thickness
            + peel_ply_thickness,
            RED,
        )

        self.play(
            ReplacementTransform(
                peel_ply,
                peel_layer,
            ),
            run_time=2,
        )

        # self.play(
        #     Rotate(
        #         peel_layer,
        #         angle=TAU,
        #         axis=OUT,
        #         about_point=center,
        #     ),
        #     run_time=2,
        # )

        ###############################################################
        # Center camera on mandrel 
        ###############################################################

        self.move_camera(
            phi=65 * DEGREES,
            theta=-90 * DEGREES,
            zoom=1.5,
        )

        # move all objects right by 2
        self.play(
            sheets.animate.shift(RIGHT*2),
            mandrel.animate.shift(RIGHT*2),
            mold_layer.animate.shift(RIGHT*2),
            glass_layer.animate.shift(RIGHT*2),
            peel_layer.animate.shift(RIGHT*2)
        )

        ###############################################################
        # Remove mandrel
        ###############################################################

        self.play(
            mandrel.animate.shift(OUT*4).set_opacity(0)
        )

        ###############################################################
        # Remove consumables
        ###############################################################

        self.play(
            mold_layer.animate.shift(OUT*4).set_opacity(0),
            peel_layer.animate.shift(OUT*4).set_opacity(0)
        )

        ###############################################################
        # Finished fiberglass tube
        ###############################################################

        self.wait(2)

Manim Community v0.20.1

[08/04/26 12:47:30] WARNING  It looks like the scene contains a lot of sub-mobjects. Caching is      ]8;id=3553708;file://c:\Users\sampu\OneDrive\Documents\GitHub\CPSC-coursework\.venv\Lib\site-packages\manim\utils\hashing.py\hashing.py]8;;\:]8;id=3553709;file://c:\Users\sampu\OneDrive\Documents\GitHub\CPSC-coursework\.venv\Lib\site-packages\manim\utils\hashing.py#161\161]8;;\
                             sometimes not suited to handle such large scenes, you might consider                  
                             disabling caching with --disable_caching to potentially speed up the                  
                             rendering process.                                                                    

                    WARNING  You can disable this warning by setting disable_caching_warning to True ]8;id=3553714;file://c:\Users\sampu\OneDrive\Documents\GitHub\CPSC-coursework\.venv\Lib\site-packages\manim\utils\hashing.py\hashing.py]8;;\:]8;id=3553715;file://c:\Users\sampu\OneDrive\Documents\GitHub\CPSC-coursework\.venv\Lib\site-packages\manim\utils\hashing.py#167\167]8;;\
                             in your config file.                                                                  

[08/04/26 12:48:52] WARNING  It looks like the scene contains a lot of sub-mobjects. Caching is      ]8;id=3553720;file://c:\Users\sampu\OneDrive\Documents\GitHub\CPSC-coursework\.venv\Lib\site-packages\manim\utils\hashing.py\hashing.py]8;;\:]8;id=3553721;file://c:\Users\sampu\OneDrive\Documents\GitHub\CPSC-coursework\.venv\Lib\site-packages\manim\utils\hashing.py#161\161]8;;\
                             sometimes not suited to handle such large scenes, you might consider                  
                             disabling caching with --disable_caching to potentially speed up the                  
                             rendering process.                                                                    

                    WARNING  You can disable this warning by setting disable_caching_warning to True ]8;id=3553726;file://c:\Users\sampu\OneDrive\Documents\GitHub\CPSC-coursework\.venv\Lib\site-packages\manim\utils\hashing.py\hashing.py]8;;\:]8;id=3553727;file://c:\Users\sampu\OneDrive\Documents\GitHub\CPSC-coursework\.venv\Lib\site-packages\manim\utils\hashing.py#167\167]8;;\
                             in your config file.                                                                  

[08/04/26 12:50:04] WARNING  It looks like the scene contains a lot of sub-mobjects. Caching is      ]8;id=3553732;file://c:\Users\sampu\OneDrive\Documents\GitHub\CPSC-coursework\.venv\Lib\site-packages\manim\utils\hashing.py\hashing.py]8;;\:]8;id=3553733;file://c:\Users\sampu\OneDrive\Documents\GitHub\CPSC-coursework\.venv\Lib\site-packages\manim\utils\hashing.py#161\161]8;;\
                             sometimes not suited to handle such large scenes, you might consider                  
                             disabling caching with --disable_caching to potentially speed up the                  
                             rendering process.                                                                    

                    WARNING  You can disable this warning by setting disable_caching_warning to True ]8;id=3553738;file://c:\Users\sampu\OneDrive\Documents\GitHub\CPSC-coursework\.venv\Lib\site-packages\manim\utils\hashing.py\hashing.py]8;;\:]8;id=3553739;file://c:\Users\sampu\OneDrive\Documents\GitHub\CPSC-coursework\.venv\Lib\site-packages\manim\utils\hashing.py#167\167]8;;\
                             in your config file.                                                                  

[08/04/26 12:51:17] WARNING  It looks like the scene contains a lot of sub-mobjects. Caching is      ]8;id=3553744;file://c:\Users\sampu\OneDrive\Documents\GitHub\CPSC-coursework\.venv\Lib\site-packages\manim\utils\hashing.py\hashing.py]8;;\:]8;id=3553745;file://c:\Users\sampu\OneDrive\Documents\GitHub\CPSC-coursework\.venv\Lib\site-packages\manim\utils\hashing.py#161\161]8;;\
                             sometimes not suited to handle such large scenes, you might consider                  
                             disabling caching with --disable_caching to potentially speed up the                  
                             rendering process.                                                                    

                    WARNING  You can disable this warning by setting disable_caching_warning to True ]8;id=3553750;file://c:\Users\sampu\OneDrive\Documents\GitHub\CPSC-coursework\.venv\Lib\site-packages\manim\utils\hashing.py\hashing.py]8;;\:]8;id=3553751;file://c:\Users\sampu\OneDrive\Documents\GitHub\CPSC-coursework\.venv\Lib\site-packages\manim\utils\hashing.py#167\167]8;;\
                             in your config file.                                                                  